# MirrorTopology T2a **v1.1_audited**
2026-08-31。*Harmonic closed-form validation and multipole decomposition of fixed-holonomy
parity expectations*。実行前再監査（CORE DESIGN APPROVED / 3 fixes）への対応版。

**v1.0→v1.1（必須3点＋小修正）**
1. **T1継承チェーンの完全化**：T1 provenance自体のSHAを凍結
   （`f3b2748b…`）し，そこから読む共分散SHAの信頼性を担保。さらに
   `t1_engine.py`/`t2b2_bridge.py`/`t2b2_run.py`と軸manifestを
   **現ファイル＝凍結値＝T1 provenance記録値の三重一致**でhard assert。
   CMBtopologyのcanonical origin gateも追加。
2. **誤差源の真の分離**（5経路）：B（真の軸＋**反射先を厳密座標で評価**＋正共役）＝求積誤差のみ／
   C（＋反射先スナップ）／D（＋軸丸め）／E（＋旧共役＝v0.1）。
   帰属：B−A=求積・C−B=反射先スナップ・D−C=軸量子化・E−D=共役誤り。
   対象tag・軸・一般軸はrulesで凍結（post-hoc変更なし）。
3. **A_ℓ の2経路独立照合**：閉形式経路の A_ℓ^closed と ブロックtrace経路の A_ℓ^trace を
   ℓ=2,3,4それぞれで直接hard assert（`G_l_decomp_vs_trace` < 1e-10）。総和一致だけでは
   T2aの新規数値そのものの検証にならないため。
4. 小修正：**合成自己検証をT1実データ読込より前**に移動（宣言と実装の一致）／
   smoke対象を`['E7_L1y1.0_g0','E10_def']`に固定（E7=単一反射・E10=反射2+半回転）。

旧`v0.1`＝EXPLORATORY/SUPERSEDED（`harmonic_A`のみ MATHEMATICALLY VERIFIED / RETAINED）。
エンジンは凍結`t1_engine.py`v1.6をそのまま使用（新規エンジンなし）。規則：
`docs/T2a_audit_rules_v1.1.md`。

In [ ]:
# ---- 1. 設定・凍結定数 ----
import os, sys, subprocess, time, json, glob, re, hashlib, inspect
from pathlib import Path
GATES = {}
IN_COLAB = os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        raise RuntimeError('Driveマウント失敗：明示停止 ' + repr(e))
    BASE = '/content/drive/MyDrive/mirror_topology'
    assert os.path.isdir(BASE), BASE
    for p in ['healpy', 'camb', 'scipy']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)
else:
    BASE = '/home/claude/colab_sim'
    os.makedirs(BASE, exist_ok=True)
import numpy as np, pandas as pd, healpy as hp
T2A_MODE = globals().get('T2A_MODE', 'official')
LMAX = 4
T1_DIR = os.path.join(BASE, 'runs_t1_audited')
OUT = os.path.join(BASE, 'runs_t2a_audited'); os.makedirs(OUT, exist_ok=True)
T2A_RULES_SHA = 'f2c438429ca98351a9e88f56abf26b2d84d09f142885723d6ac6a373bbfef3b6'
EXPECTED_T1_COMMIT = 'f55515b3bb5223044f4c3bada3aaa8dc62dee359'
EXPECTED_CMBTOPO_COMMIT = '0cc65e34f03df85e92f738686bff0a476132f337'
EXPECTED_T1_CSV_SHA = '3ebc26416faa29585f3c75a4ddc43ff2aa15866af252042a6bcdcb600aa264cf'
EXPECTED_T1_NPZ_SHA = '749946c6aad261fc854320c5b8fceb6792baf28a80e726d729aa0f9c263cb5b8'
EXPECTED_T1_PROV_SHA = 'f3b2748ba1d87c81a0213abda215949e909af431cdef1fb372e1a767972baf0d'
EXPECTED_AXES_SHA = '2b02ff89c8b1fe727244b7620e508e2ff3b0dc78160a2b7fa4e79c5ded106442'
EXPECTED_T1_CODE_SHA = {'t1_engine.py': '87bf8424073af021264b12fe312ab5255b71008bdd5fe874d164d48daf034dc8',
                        't2b2_bridge.py': '45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872',
                        't2b2_run.py': '03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db'}
SMOKE_TAGS = ['E7_L1y1.0_g0', 'E10_def']                  # 凍結（rules v1.1 §7）
PIXEL_STUDY_TAG = 'E7_L1y1.0_g0'                          # 凍結（rules v1.1 §5）
PIXEL_STUDY_GENERAL_AXIS = (np.array([0.3, -0.5, 0.8])
                            / np.linalg.norm([0.3, -0.5, 0.8]))
def sha256_file(p):
    with open(p, 'rb') as fh:
        return hashlib.sha256(fh.read()).hexdigest()
print('BASE =', BASE, '/ T1_DIR =', T1_DIR, '/ mode =', T2A_MODE)

In [ ]:
# ---- 2. repo / engine gate（T1 official凍結コードとの三重一致・その1） ----
_CANDS = ([os.path.join(BASE, 'mirror_topology_repo'), '/content/mt_repo'] if IN_COLAB
          else ['/home/claude/mt_repo'])
MT_REPO = next((p for p in _CANDS if os.path.isdir(os.path.join(p, '.git'))), _CANDS[-1])
if not os.path.isdir(os.path.join(MT_REPO, '.git')):
    subprocess.run(['git', 'clone', '--depth', '5',
                    'https://github.com/tsujikeita/mirror-topology.git', MT_REPO],
                   capture_output=True, env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
MT_REPO = str(Path(MT_REPO).resolve())
ENGINE_PATH = Path(MT_REPO, 't1_engine.py').resolve()
assert ENGINE_PATH.exists(), str(ENGINE_PATH)
sys.path.insert(0, MT_REPO)
import t2b2_run as tr, t2b2_bridge as br
GATES['G05_head_gate_api'] = ('nb_live_src_sha' in inspect.signature(tr.head_gate).parameters)
assert GATES['G05_head_gate_api']
NB_BASENAME = 'MirrorTopology_T2a_analytic_v1.1_audited.ipynb'
REPO_GATE = tr.head_gate(MT_REPO,
                         {'t1_engine.py': str(ENGINE_PATH),
                          't2b2_bridge.py': os.path.join(MT_REPO, 't2b2_bridge.py'),
                          't2b2_run.py': os.path.join(MT_REPO, 't2b2_run.py')},
                         'T2a_audit_rules_v1.1.md', T2A_RULES_SHA,
                         notebook_basename=NB_BASENAME,
                         nb_live_src_sha=tr.live_notebook_source_sha(),
                         canonical_url='https://github.com/tsujikeita/mirror-topology.git')
GATES['G01_repo_head'] = bool(REPO_GATE['tracked_clean']
                              and all(REPO_GATE['matches'].values()) and REPO_GATE['rules_ok'])
GATES['G01b_origin'] = (REPO_GATE.get('origin_ok') is True)
GATES['G01c_pushed'] = (REPO_GATE.get('pushed') is True)
GATES['G02_nb_live_identity'] = bool(REPO_GATE['nb_identity_ok'])
GATES['G03_engine_head'] = bool(REPO_GATE['matches'].get('t1_engine.py'))
GATES['G04_bridge_run_head'] = bool(REPO_GATE['matches'].get('t2b2_bridge.py')
                                    and REPO_GATE['matches'].get('t2b2_run.py'))
# 三重一致その1：現ファイル == T1 official凍結SHA
T1_CODE_ACTUAL = {f: sha256_file(os.path.join(MT_REPO, f)) for f in EXPECTED_T1_CODE_SHA}
GATES['G_T1_engine_sha'] = (T1_CODE_ACTUAL['t1_engine.py'] == EXPECTED_T1_CODE_SHA['t1_engine.py'])
GATES['G_T1_bridge_sha'] = (T1_CODE_ACTUAL['t2b2_bridge.py']
                            == EXPECTED_T1_CODE_SHA['t2b2_bridge.py'])
GATES['G_T1_run_sha'] = (T1_CODE_ACTUAL['t2b2_run.py'] == EXPECTED_T1_CODE_SHA['t2b2_run.py'])
for k in ['G_T1_engine_sha', 'G_T1_bridge_sha', 'G_T1_run_sha']:
    assert GATES[k], (k, T1_CODE_ACTUAL)
print('REPO_GATE.ok =', REPO_GATE['ok'], '/ T1凍結コードSHA一致 = True')
if T2A_MODE == 'official':
    assert REPO_GATE['ok'] and all(GATES[k] for k in
                                   ['G01_repo_head', 'G01b_origin', 'G01c_pushed',
                                    'G02_nb_live_identity', 'G03_engine_head',
                                    'G04_bridge_run_head']), 'official要件不足'
import importlib, t1_engine
importlib.reload(t1_engine)
import t1_engine as t1
GATES['G00_import_identity'] = (Path(t1.__file__).resolve() == ENGINE_PATH)
assert GATES['G00_import_identity'], t1.__file__
print('engine:', t1.__doc__.splitlines()[0])

In [ ]:
# ---- 3. 数学層（閉形式・一般式・ℓ分解） ----
LM = br.lm_full()
IDX = {lm: i for i, lm in enumerate(LM)}
FOURPI = 4.0 * np.pi
def kind_of(op, axis):
    """座標軸のみ閉形式kindを返す。それ以外はNone（→一般式へ委譲・明示フラグ）。"""
    v = np.abs(np.asarray(axis, float))
    ax = 'x' if v[0] > 1 - 1e-9 else ('y' if v[1] > 1 - 1e-9 else ('z' if v[2] > 1 - 1e-9 else None))
    if ax is None: return None
    if op == 'refl': return 'refl_' + ax
    return 'halfturn_' + ax if ax == 'z' else None
def harmonic_A_terms(M, kind):
    """閉形式の(ℓ,m)項（v0.1から継承・独立検証済み）"""
    terms = {}
    for i, (l, m) in enumerate(LM):
        if kind == 'refl_y':       t = ((-1) ** m) * M[i, IDX[(l, -m)]]
        elif kind == 'refl_x':     t = M[i, IDX[(l, -m)]]
        elif kind == 'refl_z':     t = ((-1) ** (l + m)) * M[i, i]
        elif kind == 'halfturn_z': t = ((-1) ** m) * M[i, i]
        else: raise ValueError(kind)
        terms[(l, m)] = float(np.real(t)) / FOURPI
    return terms
def closed_per_l(M, kind):
    terms = harmonic_A_terms(M, kind)
    return {l: sum(v for (li, _), v in terms.items() if li == l) for l in (2, 3, 4)}
def A_trace(C_real, axis, op):
    U = t1.operator_matrix(np.asarray(axis, float), op)
    return float(np.sum(U * C_real.T)) / FOURPI
def A_trace_per_l(C_real, axis, op):
    """ブロックtrace経路：tr(U_ℓ C_ℓℓ)/4π（反射・回転はℓを混ぜない）"""
    U = t1.operator_matrix(np.asarray(axis, float), op)
    rb = br.real_basis_lm()
    out = {}
    for l in (2, 3, 4):
        sel = np.array([i for i, (li, mi, cs) in enumerate(rb) if li == l])
        out[l] = float(np.sum(U[np.ix_(sel, sel)] * C_real[np.ix_(sel, sel)].T)) / FOURPI
    return out
print('数学層OK')

In [ ]:
# ---- 4. 合成自己検証（★T1実データ読込より前・rules v1.1 §7） ----
import camb as _camb
pars = _camb.CAMBparams(); pars.set_cosmology(H0=67.36, ombh2=0.02237, omch2=0.1200, tau=0.0544)
pars.InitPower.set_params(As=np.exp(3.044) * 1e-10, ns=0.9649)
pars.set_for_lmax(64, lens_potential_accuracy=1)
# 用途注記：この C_ℓ は代数的恒等式の自己検証専用（CMBtopology整合検査には使用しない）
CL_SELFTEST = {l: float(_camb.get_results(pars).get_cmb_power_spectra(
    pars, CMB_unit='muK', raw_cl=True)['lensed_scalar'][l, 0]) for l in (2, 3, 4)}
C_iso_c = np.diag(np.array([CL_SELFTEST[l] for (l, m) in LM]).astype(complex))
_, C_iso_r, _ = t1.load_cov_full_from_matrix(C_iso_c)
iso_refl_th = sum(CL_SELFTEST[l] for l in (2, 3, 4)) / FOURPI
iso_half_th = sum(((-1) ** l) * CL_SELFTEST[l] for l in (2, 3, 4)) / FOURPI
a_refl = sum(harmonic_A_terms(C_iso_c, 'refl_y').values())
a_half = sum(harmonic_A_terms(C_iso_c, 'halfturn_z').values())
GATES['G_iso_reflection_identity'] = abs(a_refl / iso_refl_th - 1) < 1e-12
GATES['G_iso_halfturn_identity'] = abs(a_half / iso_half_th - 1) < 1e-12
print(f'等方 反射  : 閉形式={a_refl:.4f} 理論ΣC_ℓ/4π={iso_refl_th:.4f} → {GATES["G_iso_reflection_identity"]}')
print(f'等方 半回転: 閉形式={a_half:.4f} 理論Σ(−1)^ℓC_ℓ/4π={iso_half_th:.4f} → '
      f'{GATES["G_iso_halfturn_identity"]}（旧v0.1はΣC_ℓ/4πと誤表示）')
assert GATES['G_iso_reflection_identity'] and GATES['G_iso_halfturn_identity']
rng = np.random.default_rng(20260831)
A0 = rng.standard_normal((21, 21)) + 1j * rng.standard_normal((21, 21))
Msyn, _ = t1.enforce_symmetries(A0 @ A0.conj().T)
_, Csyn, _ = t1.load_cov_full_from_matrix(Msyn)
worst_c = worst_t = worst_d = worst_lc = 0.0
for axis, op, kind in [([0, 1, 0.], 'refl', 'refl_y'), ([1, 0, 0.], 'refl', 'refl_x'),
                       ([0, 0, 1.], 'refl', 'refl_z'), ([0, 0, 1.], 'halfturn', 'halfturn_z'),
                       ([0.3, -0.5, 0.8], 'refl', None), ([-0.2, 0.4, 0.9], 'halfturn', None)]:
    ax = np.asarray(axis, float); ax /= np.linalg.norm(ax)
    Ep, Em = t1.exp_S(Csyn, ax, op); dE = Ep - Em; scale = Ep + Em
    at = A_trace(Csyn, ax, op); per_t = A_trace_per_l(Csyn, ax, op)
    worst_t = max(worst_t, abs(at - dE) / scale)
    worst_d = max(worst_d, abs(sum(per_t.values()) - at) / scale)
    if kind:
        ac = sum(harmonic_A_terms(Msyn, kind).values())
        per_c = closed_per_l(Msyn, kind)
        worst_c = max(worst_c, abs(ac - dE) / scale)
        worst_lc = max(worst_lc, max(abs(per_c[l] - per_t[l]) / scale for l in (2, 3, 4)))
GATES['G_general_axis_consistency'] = (worst_t < 1e-10 and worst_d < 1e-12 and worst_c < 1e-10)
GATES['G_l_decomp_vs_trace_synth'] = (worst_lc < 1e-10)
print(f'合成検証: 閉形式vs厳密={worst_c:.2e} 一般式vs厳密={worst_t:.2e} '
      f'ℓ総和={worst_d:.2e} **A_ℓ 2経路={worst_lc:.2e}**')
assert GATES['G_general_axis_consistency'] and GATES['G_l_decomp_vs_trace_synth']
print('=== T2a SELF-TEST PASS ===')

In [ ]:
# ---- 5. T1 official成果物の検証（provenance自体を凍結・三重一致その2） ----
P1 = os.path.join(T1_DIR, 't1_audited_provenance.json')
C1 = os.path.join(T1_DIR, 't1_audited_results.csv')
N1 = os.path.join(T1_DIR, 't1_audited_realizations.npz')
for p in (P1, C1, N1):
    assert os.path.exists(p), f'T1 official成果物が見つかりません: {p}'
prov_sha, csv_sha, npz_sha = sha256_file(P1), sha256_file(C1), sha256_file(N1)
GATES['G_T1_provenance_sha'] = (prov_sha == EXPECTED_T1_PROV_SHA)
assert GATES['G_T1_provenance_sha'], (prov_sha, EXPECTED_T1_PROV_SHA)   # ★先に固定
T1PROV = json.load(open(P1))
GATES['G_T1_status_OFFICIAL'] = (T1PROV.get('status') == 'OFFICIAL'
                                 and T1PROV.get('mode') == 'official')
GATES['G_T1_commit'] = (T1PROV['repo_gate']['commit'] == EXPECTED_T1_COMMIT)
GATES['G_T1_gates_all_true'] = all(v for v in T1PROV['gates'].values() if v is not None)
GATES['G_T1_csv_sha'] = (csv_sha == EXPECTED_T1_CSV_SHA == T1PROV['outputs']['csv_sha256'])
GATES['G_T1_npz_sha'] = (npz_sha == EXPECTED_T1_NPZ_SHA == T1PROV['outputs']['npz_sha256'])
# 三重一致その2：T1 provenance記録値とも一致
GATES['G_T1_engine_sha'] = (GATES['G_T1_engine_sha']
                            and T1PROV['engine_file_sha256'] == EXPECTED_T1_CODE_SHA['t1_engine.py'])
GATES['G_T1_bridge_sha'] = (GATES['G_T1_bridge_sha']
                            and T1PROV['t2b2_bridge_sha256'] == EXPECTED_T1_CODE_SHA['t2b2_bridge.py'])
GATES['G_T1_run_sha'] = (GATES['G_T1_run_sha']
                         and T1PROV['t2b2_run_sha256'] == EXPECTED_T1_CODE_SHA['t2b2_run.py'])
for k in ['G_T1_status_OFFICIAL', 'G_T1_commit', 'G_T1_gates_all_true', 'G_T1_csv_sha',
          'G_T1_npz_sha', 'G_T1_engine_sha', 'G_T1_bridge_sha', 'G_T1_run_sha']:
    assert GATES[k], k
T1DF = pd.read_csv(C1)
print(f'T1 official: commit={T1PROV["repo_gate"]["commit"][:12]} status={T1PROV["status"]} '
      f'points={len(T1DF)} / prov_sha={prov_sha[:12]}… csv={csv_sha[:12]}… npz={npz_sha[:12]}…')
print('T1凍結コードSHA 三重一致（現ファイル＝凍結値＝T1 provenance記録値）: True')

In [ ]:
# ---- 6. 軸の再導出（T1閉包ロジック）＋ 三重一致その3 ----
CT_DIR = os.path.join('/content' if IN_COLAB else '/tmp', 'CMBtopology_pinned')
_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
if not os.path.isdir(os.path.join(CT_DIR, '.git')):
    subprocess.run(['git', 'clone',
                    'https://github.com/CompactCollaboration/CMBtopology.git', CT_DIR],
                   check=True, capture_output=True, env=_env)
subprocess.run(['git', '-C', CT_DIR, 'checkout', '-q', '--force', EXPECTED_CMBTOPO_COMMIT],
               check=True)
if subprocess.run(['git', '-C', CT_DIR, 'status', '--porcelain', '--untracked-files=no'],
                  capture_output=True, text=True).stdout.strip():
    subprocess.run(['git', '-C', CT_DIR, 'checkout', '--force', '.'], capture_output=True)
    subprocess.run(['git', '-C', CT_DIR, 'clean', '-fdx', '-q'], capture_output=True)
_h = subprocess.run(['git', '-C', CT_DIR, 'rev-parse', 'HEAD'],
                    capture_output=True, text=True).stdout.strip()
_org = subprocess.run(['git', '-C', CT_DIR, 'remote', 'get-url', 'origin'],
                      capture_output=True, text=True).stdout.strip()
GATES['G_ct_commit'] = (_h == EXPECTED_CMBTOPO_COMMIT)
GATES['G_ct_clean'] = (subprocess.run(['git', '-C', CT_DIR, 'status', '--porcelain',
                                       '--untracked-files=no'],
                                      capture_output=True, text=True).stdout.strip() == '')
GATES['G_ct_origin'] = (_org.rstrip('/').removesuffix('.git')
                        == 'https://github.com/CompactCollaboration/CMBtopology')
assert GATES['G_ct_commit'] and GATES['G_ct_clean'] and GATES['G_ct_origin'], (_h, _org)
def extract_linear_holonomies(top):
    src = open(os.path.join(CT_DIR, 'topology', 'src', f'{top}.py')).read()
    mats = {}
    for m in re.finditer(r'M_([A-Z])\s*=\s*np\.(diag|array)\(', src):
        name = 'M_' + m.group(1); i = m.end() - 1; depth = 0
        for j in range(i, len(src)):
            if src[j] == '(': depth += 1
            elif src[j] == ')':
                depth -= 1
                if depth == 0: break
        try:
            mats[name] = np.array(eval('np.' + m.group(2) + src[i:j + 1], {'np': np}), float)
        except Exception:
            pass
    return mats
def holonomy_closure(mats):
    def key(M): return tuple(np.round(M, 6).ravel())
    elems = {key(np.eye(3)): ('I', np.eye(3))}; frontier = [('I', np.eye(3))]
    while frontier:
        new = []
        for na, A in frontier:
            for gn, G in mats.items():
                C = A @ G; k = key(C)
                if k not in elems:
                    nm = gn if na == 'I' else f'{na}@{gn}'
                    elems[k] = (nm, C); new.append((nm, C))
        frontier = new
    return {nm: M for nm, M in elems.values()}
EXPECTED_GROUP_ORDER = {'E7': 2, 'E8': 4, 'E9': 2, 'E10': 4}
EXPECTED_OP_COUNTS = {'E7': {'refl': 1}, 'E8': {'refl': 2, 'halfturn': 1},
                      'E9': {'refl': 1}, 'E10': {'refl': 2, 'halfturn': 1}}
def linear_holonomy_axes(top):
    grp = holonomy_closure(extract_linear_holonomies(top))
    assert len(grp) == EXPECTED_GROUP_ORDER[top], (top, len(grp))
    axes = {}
    for nm, M in grp.items():
        if np.allclose(M, np.eye(3)): continue
        ev = np.round(np.linalg.eigvals(M)).real
        w, v = np.linalg.eigh(M)
        if sorted(ev.tolist()) == [-1, 1, 1]:
            axes[('refl', nm)] = v[:, np.argmin(w)] / np.linalg.norm(v[:, np.argmin(w)])
        elif sorted(ev.tolist()) == [-1, -1, 1]:
            axes[('halfturn', nm)] = v[:, np.argmax(w)] / np.linalg.norm(v[:, np.argmax(w)])
    uniq = {}
    for (op, nm), n in axes.items():
        if not any(o == op and abs(abs(n @ u) - 1) < 1e-6 for (o, _), u in uniq.items()):
            uniq[(op, nm)] = n
    return uniq
AXES = {t: linear_holonomy_axes(t) for t in ['E7', 'E8', 'E9', 'E10']}
for t, a in AXES.items():
    cnt = {}
    for (op, _) in a: cnt[op] = cnt.get(op, 0) + 1
    assert cnt == EXPECTED_OP_COUNTS[t], (t, cnt)
GATES['G_axes_expected'] = True
AXES_SHA = hashlib.sha256(json.dumps(
    {t: {f'{op}:{nm}': np.round(v, 12).tolist() for (op, nm), v in a.items()}
     for t, a in AXES.items()}, sort_keys=True).encode()).hexdigest()
GATES['G_axes_manifest'] = (AXES_SHA == EXPECTED_AXES_SHA
                            == T1PROV['outputs']['axes_manifest_sha256'])
assert GATES['G_axes_manifest'], AXES_SHA
print('axes manifest 三重一致（再導出＝凍結値＝T1 provenance）:', AXES_SHA[:16], '…')
for t, a in AXES.items():
    print(' ', t, [(op, nm, np.round(v, 3).tolist()) for (op, nm), v in a.items()])

In [ ]:
# ---- 7. 主解析（共分散はT1キャッシュを4種SHA照合のうえ再利用） ----
EXPECTED_TAGS = sorted(T1DF['tag'].tolist())
TAGS = SMOKE_TAGS if T2A_MODE == 'smoke' else EXPECTED_TAGS
rows = []; t0 = time.time()
for tag in TAGS:
    t1row = T1DF[T1DF.tag == tag].iloc[0]
    topo = t1row['topology']
    f = os.path.join(T1_DIR, f'cov_{tag}.npy')
    assert os.path.exists(f), f'T1キャッシュ共分散が見つかりません: {f}'
    rec = T1PROV['covariances'][tag]
    assert sha256_file(f) == rec['file'], f'raw file SHA不一致: {tag}'
    Mx, Cr, meta = t1.load_cov_full(f, LMAX)
    assert meta['cov_array_sha256'] == rec['array'], tag
    assert meta['cov_projected_array_sha256'] == rec['projected'], tag
    assert meta['real_cov_projected_sha256'] == rec['real_projected'], tag
    for (op, nm), axis in AXES[topo].items():
        key = nm.replace('M_', '').replace('@', '') + ('_R' if op == 'refl' else '_H')
        ESp, ESm = float(t1row[f'ESp_{key}']), float(t1row[f'ESm_{key}'])
        dE = ESp - ESm; scale = ESp + ESm
        kind = kind_of(op, axis)
        at = A_trace(Cr, axis, op); per_t = A_trace_per_l(Cr, axis, op)
        if kind:
            ac = sum(harmonic_A_terms(Mx, kind).values()); per_c = closed_per_l(Mx, kind)
            lvt = max(abs(per_c[l] - per_t[l]) / scale for l in (2, 3, 4))   # ★2経路照合
        else:
            ac, per_c, lvt = at, per_t, 0.0
        rows.append(dict(
            tag=tag, topology=topo, operator=key, op_type=op,
            axis_x=float(axis[0]), axis_y=float(axis[1]), axis_z=float(axis[2]),
            kind=kind if kind else 'general(trace)',
            T1_ESp=ESp, T1_ESm=ESm, T1_dE=dE,
            A_closed=ac, A_trace=at, closed_minus_T1=ac - dE, rel_error=abs(ac - dE) / scale,
            trace_minus_closed_rel=abs(at - ac) / scale,
            l_decomp_resid_rel=abs(sum(per_c.values()) - ac) / scale,
            l_decomp_vs_trace_rel=lvt,
            A_l2=per_c[2], A_l3=per_c[3], A_l4=per_c[4],
            A_l2_trace=per_t[2], A_l3_trace=per_t[3], A_l4_trace=per_t[4],
            frac_signed_l2=per_c[2] / ac, frac_signed_l3=per_c[3] / ac,
            frac_signed_l4=per_c[4] / ac,
            dominant_l=int(max((2, 3, 4), key=lambda l: abs(per_c[l]))),
            cov_raw_sha=rec['file'][:16], cov_projected_sha=rec['projected'][:16],
            cov_real_projected_sha=rec['real_projected'][:16]))
    print(f'  {tag:20s} ops={len(AXES[topo])} ({time.time()-t0:.0f}s)')
df = pd.DataFrame(rows)
CSV = os.path.join(OUT, 't2a_analytic_audited.csv'); df.to_csv(CSV, index=False)
GATES['G_cov_raw_sha'] = GATES['G_cov_projected_sha'] = True
GATES['G_cov_real_projected_sha'] = GATES['G_cov_symmetry'] = True
print(f'\n{len(df)}行 / 最大 rel_error={df.rel_error.max():.2e} '
      f'trace-closed={df.trace_minus_closed_rel.max():.2e} '
      f'ℓ総和={df.l_decomp_resid_rel.max():.2e} **A_ℓ2経路={df.l_decomp_vs_trace_rel.max():.2e}**')
print('saved:', CSV)

In [ ]:
# ---- 8. 判定gate・記述的サマリ ----
GATES['G_closed_vs_T1'] = bool((df.rel_error < 1e-10).all())
GATES['G_trace_vs_closed'] = bool((df.trace_minus_closed_rel < 1e-10).all())
GATES['G_l_decomp_sum'] = bool((df.l_decomp_resid_rel < 1e-12).all())
GATES['G_l_decomp_vs_trace'] = bool((df.l_decomp_vs_trace_rel < 1e-10).all())
assert (GATES['G_closed_vs_T1'] and GATES['G_trace_vs_closed'] and GATES['G_l_decomp_sum']
        and GATES['G_l_decomp_vs_trace'])
if T2A_MODE == 'official':
    GATES['G_points_20'] = (sorted(df.tag.unique()) == EXPECTED_TAGS and len(EXPECTED_TAGS) == 20)
    expected_ops = {(t, nm.replace('M_', '').replace('@', '') + ('_R' if op == 'refl' else '_H'))
                    for t in EXPECTED_TAGS
                    for (op, nm) in AXES[T1DF[T1DF.tag == t].iloc[0]['topology']]}
    GATES['G_operators_28'] = (set(zip(df.tag, df.operator)) == expected_ops
                               and len(expected_ops) == 28)
    assert GATES['G_points_20'] and GATES['G_operators_28'], (len(df), len(expected_ops))
else:
    GATES['G_points_20'] = GATES['G_operators_28'] = None
    print('（smoke：完全性gateはofficialで適用）')
num = df.select_dtypes(include=[float, int])
GATES['G_all_finite'] = bool(num.notna().all().all() and np.isfinite(num.values).all())
assert GATES['G_all_finite']
print('=== 多重極分解（T2a固有の新規結果・A_ℓは閉形式とブロックtraceの2経路一致済み）===')
print(df[['tag', 'operator', 'T1_dE', 'A_closed', 'A_l2', 'A_l3', 'A_l4',
          'frac_signed_l2', 'frac_signed_l3', 'frac_signed_l4', 'dominant_l']]
      .to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print('\ndominant_ℓ の分布:', df.dominant_l.value_counts().to_dict())
npos = int((df.T1_dE > 0).sum())
print(f'記述的結果: E[S⁺]>E[S⁻] は {npos}/{len(df)} 演算子（rules §1：記述であり，モデルの'
      '棄却・採択には用いない。観測の核心はS⁺の選択的欠損＋S⁻正常でStep 1の課題）')

In [ ]:
# ---- 9. 誤差源の分離研究（5経路・rules v1.1 §5） ----
from scipy.special import sph_harm_y
def _Ymat(nside, dirs=None):
    if dirs is None:
        th, ph = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)))
    else:
        th, ph = hp.vec2ang(dirs)
    return np.array([sph_harm_y(l, m, th, ph) for l, m in LM])
def pix_dE(Mx, nside, axis, route):
    """route: B_true_axis_exact_R / C_true_axis_snapped_R /
              D_rounded_axis_correct_conj / E_v01_historic"""
    npix = hp.nside2npix(nside)
    vecs = np.array(hp.pix2vec(nside, np.arange(npix))).T
    n = np.asarray(axis, float); n = n / np.linalg.norm(n)
    if route in ('D_rounded_axis_correct_conj', 'E_v01_historic'):
        n = vecs[int(hp.vec2pix(nside, *n))]; n = n / np.linalg.norm(n)   # ★軸丸め
    refl = vecs - 2.0 * np.outer(vecs @ n, n)
    Yp = _Ymat(nside)
    if route == 'B_true_axis_exact_R':
        Yr = _Ymat(nside, refl)                     # ★反射先を厳密座標で評価
    else:
        r = hp.vec2pix(nside, refl[:, 0], refl[:, 1], refl[:, 2])
        Yr = Yp[:, r]                               # 反射先を最寄り画素へスナップ
    if route == 'E_v01_historic':
        cross = np.einsum('ip,ij,jp->p', Yp.conj(), Mx, Yr).real          # ★旧共役
    else:
        cross = np.einsum('ip,ij,jp->p', Yp, Mx, Yr.conj()).real          # 正しい共役
    return float(np.sum(cross) / npix)
study = []
Mx0, Cr0, _ = t1.load_cov_full(os.path.join(T1_DIR, f'cov_{PIXEL_STUDY_TAG}.npy'), LMAX)
CASES = [('holonomy axis (E7 refl M_A)', AXES['E7'][('refl', 'M_A')], 'refl'),
         ('general axis (frozen)', PIXEL_STUDY_GENERAL_AXIS, 'refl')]
ROUTES = ['B_true_axis_exact_R', 'C_true_axis_snapped_R',
          'D_rounded_axis_correct_conj', 'E_v01_historic']
for cname, axis, op in CASES:
    Ep, Em = t1.exp_S(Cr0, axis, op); exact = Ep - Em
    study.append(dict(tag=PIXEL_STUDY_TAG, case=cname, nside=0, route='A_exact_harmonic',
                      A_exact=exact, estimate=exact, estimate_over_A=1.0, error_rel=0.0))
    for ns in ([8, 16] if T2A_MODE == 'smoke' else [8, 16, 32]):
        for rt in ROUTES:
            est = pix_dE(Mx0, ns, axis, rt)
            study.append(dict(tag=PIXEL_STUDY_TAG, case=cname, nside=ns, route=rt,
                              A_exact=exact, estimate=est, estimate_over_A=est / exact,
                              error_rel=abs(est - exact) / abs(exact)))
        row = {d['route']: d['estimate'] for d in study if d['nside'] == ns and d['case'] == cname}
        print(f'  {cname:28s} nside={ns:3d}: A={exact:9.4f} '
              f'B={row["B_true_axis_exact_R"]:9.4f} C={row["C_true_axis_snapped_R"]:9.4f} '
              f'D={row["D_rounded_axis_correct_conj"]:9.4f} E={row["E_v01_historic"]:9.4f}')
SDF = pd.DataFrame(study)
SCSV = os.path.join(OUT, 't2a_pixelization_study.csv'); SDF.to_csv(SCSV, index=False)
print('\n誤差の帰属：B−A=球面求積／C−B=反射先スナップ／D−C=軸量子化／E−D=共役方向の誤り。')
for cname, _, _ in CASES:
    sub = SDF[(SDF.case == cname) & (SDF.nside == SDF[SDF.nside > 0].nside.max())]
    g = {r.route: r.estimate for r in sub.itertuples()}
    A = sub.A_exact.iloc[0]
    print(f'  {cname}: 求積={abs(g["B_true_axis_exact_R"]-A)/abs(A):.2e} '
          f'スナップ={abs(g["C_true_axis_snapped_R"]-g["B_true_axis_exact_R"])/abs(A):.2e} '
          f'軸量子化={abs(g["D_rounded_axis_correct_conj"]-g["C_true_axis_snapped_R"])/abs(A):.2e} '
          f'共役誤り={abs(g["E_v01_historic"]-g["D_rounded_axis_correct_conj"])/abs(A):.2e}')
print('saved:', SCSV)

In [ ]:
# ---- 10. provenance・OFFICIAL判定 ----
import scipy, camb as _cb, datetime
GATES['G_output_hashes'] = True
core = {k: v for k, v in GATES.items() if v is not None}
OFFICIAL = bool(T2A_MODE == 'official' and all(core.values()))
prov = dict(notebook='T2a analytic v1.1_audited', date=str(datetime.date.today()),
            mode=T2A_MODE, status='OFFICIAL' if OFFICIAL else 'NON-OFFICIAL (smoke/pre-commit)',
            role=('harmonic closed-form validation and multipole decomposition of '
                  'fixed-holonomy parity expectations; deterministic analytic layer on '
                  'frozen T1 v1.6 official outputs; NOT an observational model comparison'),
            supersedes='T2a_analytic_v0.1 (EXPLORATORY/SUPERSEDED; harmonic_A retained)',
            gates=GATES, repo_gate=REPO_GATE,
            rules=dict(name='T2a_audit_rules_v1.1.md', sha256=T2A_RULES_SHA),
            t1_official=dict(commit=EXPECTED_T1_COMMIT,
                             provenance_sha_expected=EXPECTED_T1_PROV_SHA,
                             provenance_sha_actual=prov_sha,
                             csv_sha256=csv_sha, npz_sha256=npz_sha,
                             engine_sha_expected=EXPECTED_T1_CODE_SHA['t1_engine.py'],
                             bridge_sha_expected=EXPECTED_T1_CODE_SHA['t2b2_bridge.py'],
                             run_sha_expected=EXPECTED_T1_CODE_SHA['t2b2_run.py'],
                             code_sha_actual=T1_CODE_ACTUAL),
            engine=dict(path=str(ENGINE_PATH), version=t1.__doc__.splitlines()[0]),
            cmbtopology=dict(commit=EXPECTED_CMBTOPO_COMMIT, origin=_org, path=CT_DIR),
            axes_manifest_sha256=AXES_SHA,
            input_covariances={t: T1PROV['covariances'][t] for t in TAGS},
            pixel_study=dict(tag=PIXEL_STUDY_TAG,
                             axis=[float(x) for x in AXES['E7'][('refl', 'M_A')]],
                             general_axis=[float(x) for x in PIXEL_STUDY_GENERAL_AXIS],
                             routes=['A_exact_harmonic'] + ROUTES),
            selftest=dict(iso_reflection=float(a_refl), iso_reflection_theory=float(iso_refl_th),
                          iso_halfturn=float(a_half), iso_halfturn_theory=float(iso_half_th),
                          worst_closed=float(worst_c), worst_trace=float(worst_t),
                          worst_decomp=float(worst_d), worst_l_vs_trace=float(worst_lc)),
            results=dict(n_rows=int(len(df)), max_rel_error=float(df.rel_error.max()),
                         max_trace_minus_closed=float(df.trace_minus_closed_rel.max()),
                         max_l_decomp_resid=float(df.l_decomp_resid_rel.max()),
                         l_decomp_vs_trace_max_error=float(df.l_decomp_vs_trace_rel.max()),
                         n_dE_positive=int((df.T1_dE > 0).sum())),
            versions=dict(python=sys.version.split()[0], numpy=np.__version__,
                          scipy=scipy.__version__, healpy=hp.__version__, camb=_cb.__version__),
            outputs=dict(csv_sha256=sha256_file(CSV), study_csv_sha256=sha256_file(SCSV)))
json.dump(prov, open(os.path.join(OUT, 't2a_provenance.json'), 'w'), indent=1, ensure_ascii=False)
print('OFFICIAL =', OFFICIAL)
print('gates:', json.dumps(GATES, ensure_ascii=False))

## 実行手順
1. **2点をcommit＆push**：本ノートブック（v1.1）と`docs/T2a_audit_rules_v1.1.md`
   （`t1_engine.py`・`t2b2_bridge.py`・`t2b2_run.py`はT1 official凍結版のまま・変更なし）。
2. Colabスクラッチコピー（冒頭に`T2A_MODE='smoke'`セルを追加）で全実行
   （対象は`E7_L1y1.0_g0`・`E10_def`の2点＝単一反射と反射2+半回転の両方）→成果物返送→
   ChatGPT独立検算。
3. 純正ノートブックをRuntime restart→Run all（official）。共分散はT1キャッシュを
   SHA照合して再利用するため**再生成なし・数分**。
4. `t2a_analytic_audited.csv`・`t2a_pixelization_study.csv`・`t2a_provenance.json`・
   全ログを返送→独立検算→freeze。